# Day 4 · Exercise 1: Naive JSON Extraction

**What you'll build:** `extract_json` — ask the local model for JSON and parse it with `json.loads()`.

**Why it matters:** This is the baseline. Most people start here. It works maybe 70–80% of the time — which is exactly dangerous enough to seem fine in testing and break in production. Feeling this limitation firsthand is the point of this exercise.

## Your Implementation

In [ ]:
import ollama
import json

MODEL = "llama3.2"

def extract_json(text: str) -> dict:
    """Extract information from text by asking the model for JSON.

    Use ollama.chat with a system prompt that asks the model to return
    the information as a JSON object. Parse the response string with
    json.loads() and return the result.

    Do NOT add format="json" — this exercise shows the naive approach
    so you can feel its limitations firsthand.

    Args:
        text: A sentence or paragraph about a person.

    Returns:
        A dict parsed from the model's JSON response.

    Raises:
        json.JSONDecodeError: if the model returns prose instead of JSON.

    Example:
        extract_json("Alice is 30 and lives in Cape Town.")
        -> {"name": "Alice", "age": 30, "city": "Cape Town"}
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

> **Note:** Check 3 may occasionally show a `JSONDecodeError`. That's the reliability problem in action — the model returned prose instead of JSON. Re-run the cell to see if it's consistent. This is exactly what you'll fix in Exercise 2.

In [ ]:
_PASS, _FAIL, _WARN = '✅', '❌', '⚠️ '

def _ollama_running():
    try:
        import urllib.request  # stdlib — no install needed
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(extract_json), 'extract_json is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama is running
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: run ollama serve')
        return
    print(f'{_PASS} Check 2/{total}: Ollama server is reachable')
    score += 1

    # Check 3: function returns something parseable
    result = None
    try:
        result = extract_json("Alice is 30 years old and lives in Cape Town.")
        assert result is not None, 'function returned None (did you forget to return?)'
        assert isinstance(result, dict), f'expected dict, got {type(result).__name__}'
        print(f'{_PASS} Check 3/{total}: returned a dict')
        score += 1
    except json.JSONDecodeError as e:
        print(f'{_WARN} Check 3/{total}: JSONDecodeError — model returned prose instead of JSON')
        print(f'   Error: {e}')
        print('   This is the reliability problem in action! This is what Exercise 2 fixes.')
        score += 1  # count as demonstrated for gate purposes
    except AssertionError as e:
        print(f'{_FAIL} Check 3/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: unexpected error — {e}')

    # Check 4: dict has at least one key (only if we got a dict)
    if isinstance(result, dict):
        try:
            assert len(result) > 0, 'dict is empty'
            print(f'{_PASS} Check 4/{total}: dict has {len(result)} key(s): {list(result.keys())}')
            score += 1
        except AssertionError as e:
            print(f'{_FAIL} Check 4/{total}: {e}')
    else:
        print(f'{_WARN} Check 4/{total}: skipped (no dict to inspect)')
        score += 1  # not the student's fault — the reliability problem

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 1 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Run `extract_json` ten times with the same input and inspect the outputs:

```python
text = "Bob is a 28-year-old software engineer in Johannesburg."
for i in range(10):
    try:
        result = extract_json(text)
        print(f"Run {i+1}: {result}")
    except json.JSONDecodeError:
        print(f"Run {i+1}: JSONDecodeError — prose returned")
```

Count how many times you get: (a) a different set of keys, (b) a JSONDecodeError, (c) the "correct" output. That inconsistency is why the naive approach is not production-ready.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama
import json

MODEL = "llama3.2"

def extract_json(text: str) -> dict:
    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an information extractor. "
                    "Return ONLY a raw JSON object with the key information. "
                    "No explanation, no markdown, no prose — just the JSON."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    raw = response["message"]["content"]
    return json.loads(raw)
```

**Why it sometimes fails:** Without `format="json"`, Ollama places no syntactic constraint on the output. The model generates whatever it predicts is most likely — sometimes that's valid JSON, sometimes it's a sentence, sometimes it's a JSON object wrapped in markdown. `json.loads()` crashes on anything that isn't valid JSON. Exercise 2 fixes this with one extra parameter.
</details>